# Titanic Passenger Class Prediction

This notebook validates the project's reusable data, feature-engineering,
SQL, and machine-learning preprocessing components. Production logic lives
under `src/`, while this notebook presents and verifies the workflow.

In [12]:
import numpy as np
import pandas as pd

from titanic_passenger_class_prediction import (
    RANDOM_STATE,
    TARGET_COLUMN,
    load_processed_data,
    load_raw_data,
)
from titanic_passenger_class_prediction.database import (
    query_database,
)
from titanic_passenger_class_prediction.features import (
    add_passenger_features,
)
from titanic_passenger_class_prediction.modeling import (
    MODEL_FEATURES,
    build_preprocessor,
)
from titanic_passenger_class_prediction.validation import (
    data_quality_report,
)

pd.set_option("display.max_columns", 50)

print("Target column:", TARGET_COLUMN)
print("Random state:", RANDOM_STATE)

Target column: Pclass
Random state: 42


## 1. Raw Data and Data Quality

In [13]:
raw_df = load_raw_data()

print("Raw dataset shape:", raw_df.shape)
raw_df.head()

Raw dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [14]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [15]:
quality_report = data_quality_report(raw_df)

quality_report

{'shape': {'rows': 891, 'columns': 12},
 'duplicate_rows': 0,
 'duplicate_passenger_ids': 0,
 'missing_values': {'Age': {'count': 177, 'percentage': 19.87},
  'Cabin': {'count': 687, 'percentage': 77.1},
  'Embarked': {'count': 2, 'percentage': 0.22}},
 'data_types': {'PassengerId': 'int64',
  'Survived': 'int64',
  'Pclass': 'int64',
  'Name': 'str',
  'Sex': 'str',
  'Age': 'float64',
  'SibSp': 'int64',
  'Parch': 'int64',
  'Ticket': 'str',
  'Fare': 'float64',
  'Cabin': 'str',
  'Embarked': 'str'},
 'numeric_columns': ['PassengerId',
  'Survived',
  'Pclass',
  'Age',
  'SibSp',
  'Parch',
  'Fare'],
 'categorical_columns': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'],
 'invalid_values': {'negative_age_rows': 0,
  'age_above_100_rows': 0,
  'negative_fare_rows': 0,
  'negative_sibsp_rows': 0,
  'negative_parch_rows': 0},
 'unexpected_categories': {'Pclass': [],
  'Survived': [],
  'Sex': [],
  'Embarked': []}}

In [16]:
missing_values = pd.DataFrame(
    {
        "missing_count": raw_df.isna().sum(),
        "missing_percentage": (
            raw_df.isna().mean() * 100
        ).round(2),
    }
)

missing_values = (
    missing_values
    .query("missing_count > 0")
    .sort_values(
        "missing_percentage",
        ascending=False,
    )
)

missing_values

,missing_count,missing_percentage
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [17]:
target_distribution = (
    raw_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis("passenger_class")
    .reset_index(name="passenger_count")
)

target_distribution["percentage"] = (
    target_distribution["passenger_count"]
    / len(raw_df)
    * 100
).round(2)

target_distribution

,passenger_class,passenger_count,percentage
0,1,216,24.24
1,2,184,20.65
2,3,491,55.11


## 2. Deterministic Feature Engineering

In [18]:
featured_preview = add_passenger_features(raw_df)

preview_columns = [
    "Name",
    "Title",
    "Surname",
    "FamilySize",
    "IsAlone",
    "FamilySizeGroup",
    "Cabin",
    "HasCabin",
    "CabinDeck",
    "CabinCount",
    "Ticket",
    "TicketPrefix",
    "Age",
    "AgeGroup",
    "Fare",
    "FareLog",
]

featured_preview[preview_columns].head(10)

,Name,Title,Surname,FamilySize,IsAlone,FamilySizeGroup,Cabin,HasCabin,CabinDeck,CabinCount,Ticket,TicketPrefix,Age,AgeGroup,Fare,FareLog
0,"Braund, Mr. Owen Harris",Mr,Braund,2,0,Small,NaN,0,Unknown,0,A/5 21171,A_5,22.0,Young Adult,7.2500,2.110213
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,Cumings,2,0,Small,C85,1,C,1,PC 17599,PC,38.0,Adult,71.2833,4.280593
2,"Heikkinen, Miss. Laina",Miss,Heikkinen,1,1,Alone,NaN,0,Unknown,0,STON/O2. 3101282,STON_O2,26.0,Young Adult,7.9250,2.188856
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,Futrelle,2,0,Small,C123,1,C,1,113803,NONE,35.0,Young Adult,53.1000,3.990834
4,"Allen, Mr. William Henry",Mr,Allen,1,1,Alone,NaN,0,Unknown,0,373450,NONE,35.0,Young Adult,8.0500,2.202765
5,"Moran, Mr. James",Mr,Moran,1,1,Alone,NaN,0,Unknown,0,330877,NONE,NaN,NaN,8.4583,2.246893
6,"McCarthy, Mr. Timothy J",Mr,McCarthy,1,1,Alone,E46,1,E,1,17463,NONE,54.0,Adult,51.8625,3.967694
7,"Palsson, Master. Gosta Leonard",Master,Palsson,5,0,Medium,NaN,0,Unknown,0,349909,NONE,2.0,Child,21.0750,3.094446
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Mrs,Johnson,3,0,Small,NaN,0,Unknown,0,347742,NONE,27.0,Young Adult,11.1333,2.495954
9,"Nasser, Mrs. Nicholas (Adele Achem)",Mrs,Nasser,2,0,Small,NaN,0,Unknown,0,237736,NONE,14.0,Teen,30.0708,3.436268


## 3. Processed Dataset

In [19]:
processed_df = load_processed_data()

print("Processed dataset shape:", processed_df.shape)
processed_df.head()

Processed dataset shape: (891, 23)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Surname,FamilySize,IsAlone,FamilySizeGroup,HasCabin,CabinDeck,CabinCount,TicketPrefix,AgeGroup,FareLog
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,Braund,2,0,Small,0,Unknown,0,A_5,Young Adult,2.110213
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,Cumings,2,0,Small,1,C,1,PC,Adult,4.280593
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,Heikkinen,1,1,Alone,0,Unknown,0,STON_O2,Young Adult,2.188856
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,Futrelle,2,0,Small,1,C,1,NONE,Young Adult,3.990834
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,Allen,1,1,Alone,0,Unknown,0,NONE,Young Adult,2.202765


In [20]:
added_columns = sorted(
    set(processed_df.columns)
    - set(raw_df.columns)
)

print("Engineered columns:")
for column in added_columns:
    print("-", column)

Engineered columns:
- AgeGroup
- CabinCount
- CabinDeck
- FamilySize
- FamilySizeGroup
- FareLog
- HasCabin
- IsAlone
- Surname
- TicketPrefix
- Title


## 4. SQLite Analytics

In [21]:
class_profile = query_database(
    """
    SELECT *
    FROM vw_class_profile
    ORDER BY passenger_class;
    """
)

class_profile

,passenger_class,passenger_count,average_age,average_fare,average_family_size,percent_travelling_alone,percent_with_cabin
0,1,216,38.23,84.15,1.77,50.46,81.48
1,2,184,29.88,20.66,1.78,56.52,8.70
2,3,491,25.14,13.68,2.01,65.99,2.44


In [22]:
embarkation_class_mix = query_database(
    """
    SELECT *
    FROM vw_embarkation_class_mix
    ORDER BY embarkation_port, passenger_class;
    """
)

embarkation_class_mix

,embarkation_port,passenger_class,passenger_count,percentage_within_port
0,C,1,85,50.60
1,C,2,17,10.12
2,C,3,66,39.29
3,Q,1,2,2.60
4,Q,2,3,3.90
5,Q,3,72,93.51
6,S,1,127,19.72
7,S,2,164,25.47
8,S,3,353,54.81


## 5. Machine Learning Preprocessing

In [23]:
X = processed_df[MODEL_FEATURES]
y = processed_df[TARGET_COLUMN]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature matrix shape: (891, 15)
Target shape: (891,)


,Age,SibSp,Parch,Fare,FamilySize,IsAlone,HasCabin,CabinCount,FareLog,Sex,Embarked,Title,FamilySizeGroup,CabinDeck,TicketPrefix
0,22.0,1,0,7.2500,2,0,0,0,2.110213,male,S,Mr,Small,Unknown,A_5
1,38.0,1,0,71.2833,2,0,1,1,4.280593,female,C,Mrs,Small,C,PC
2,26.0,0,0,7.9250,1,1,0,0,2.188856,female,S,Miss,Alone,Unknown,STON_O2
3,35.0,1,0,53.1000,2,0,1,1,3.990834,female,S,Mrs,Small,C,NONE
4,35.0,0,0,8.0500,1,1,0,0,2.202765,male,S,Mr,Alone,Unknown,NONE


In [24]:
preprocessor = build_preprocessor()

transformed = preprocessor.fit_transform(X)
transformed_feature_names = (
    preprocessor.get_feature_names_out()
)

print("Input shape:", X.shape)
print("Transformed shape:", transformed.shape)
print(
    "Missing transformed values:",
    int(np.isnan(transformed).sum()),
)

Input shape: (891, 15)
Transformed shape: (891, 66)
Missing transformed values: 0


In [25]:
transformed_features_df = pd.DataFrame(
    {
        "feature_name": transformed_feature_names,
    }
)

transformed_features_df.head(30)

,feature_name
0,Age
1,SibSp
2,Parch
3,Fare
4,FamilySize
5,IsAlone
6,HasCabin
7,CabinCount
8,FareLog
9,Sex_female


In [26]:
assert raw_df.shape == (891, 12)
assert len(processed_df) == len(raw_df)
assert TARGET_COLUMN in processed_df.columns
assert transformed.shape[0] == len(processed_df)
assert np.isnan(transformed).sum() == 0

print("Notebook checks passed.")

Notebook checks passed.
